In [ ]:
# ===== DATA PREPARATION
#   segs.zip, train.csv, val.csv, test.csv, seg_metadata_multiannotator_subset.csv
import os, zipfile, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
P = '/content/drive/MyDrive/IMA_project'
S = 256   # images and masks are resized to 256 x 256

# ---- Step 1: unzip the masks ----
if not os.path.exists('/content/segs'):
    with zipfile.ZipFile(f'{P}/segs.zip') as z:
        z.extractall('/content/segs')
print("Masks:", len(os.listdir('/content/segs')))

# ---- Step 2: list the 2,394 images and download them from the ISIC Archive ----
ids = pd.concat([pd.read_csv(f'{P}/{n}.csv') for n in ['train', 'val', 'test']])['image']
ids = ids.str.replace('.JPG', '', regex=False)
ids.to_csv('/content/isic_ids.txt', index=False, header=False)
print("Image names saved:", len(ids))

if not os.path.exists('/content/ISIC-API-Image-Downloader'):
    !git clone https://github.com/kakumarabhishek/ISIC-API-Image-Downloader.git
%cd /content/ISIC-API-Image-Downloader
!python download.py --download_list /content/isic_ids.txt --output_dir /content/images --download_images
%cd /content

have = {os.path.splitext(f)[0] for _, _, fs in os.walk('/content/images') for f in fs}
missing = [i for i in ids if i not in have]
print("Images found:", len(ids) - len(missing), "| missing:", len(missing))

# ---- Step 3: pack photos, real human outlines and the official splits into ima_256.npz ----
img_path = {}
for r, _, fs in os.walk('/content/images'):
    for f in fs:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path[os.path.splitext(f)[0]] = os.path.join(r, f)
print("Photos found:", len(img_path))

# real human outlines only (never the MV/ST consensus masks)
seg = pd.read_csv(f'{P}/seg_metadata_multiannotator_subset.csv')
real = seg[~seg['annotator'].isin(['MV', 'ST'])]
groups = {k: v['seg_filename'].tolist() for k, v in real.groupby('ISIC_id')}

# official splits from the dataset authors
parts = []
for name in ['train', 'val', 'test']:
    df = pd.read_csv(f'{P}/{name}.csv')
    df['split'] = name
    parts.append(df)
meta = pd.concat(parts, ignore_index=True)
meta['id'] = meta['image'].str.replace('.JPG', '', regex=False)

no_img = [i for i in meta['id'] if i not in img_path]
no_msk = [i for i in meta['id'] if i not in groups]
print("Missing photos:", len(no_img), "| images without outlines:", len(no_msk))
assert not no_img and not no_msk, "Some files are missing"

N = len(meta)
maxm = max(len(groups[i]) for i in meta['id'])
X = np.zeros((N, S, S, 3), np.uint8)       # photos
M = np.zeros((N, maxm, S, S), np.uint8)    # human outlines (unused slots stay zero = padding)
K = np.zeros(N, np.int64)                  # how many real outlines each image has
shape_mismatch = 0

for n, iid in enumerate(tqdm(meta['id'])):
    im = Image.open(img_path[iid]).convert('RGB')
    X[n] = np.array(im.resize((S, S), Image.BILINEAR))
    for j, fn in enumerate(groups[iid]):
        m = Image.open(f'/content/segs/{fn}').convert('L')
        if abs(m.size[0] / m.size[1] - im.size[0] / im.size[1]) > 0.02:
            shape_mismatch += 1
        M[n, j] = np.array(m.resize((S, S), Image.NEAREST)) > 127
    K[n] = len(groups[iid])

np.savez_compressed(f'{P}/ima_256.npz', X=X, M=M, K=K,
    id=meta['id'].values.astype(str), split=meta['split'].values.astype(str),
    dice_range=meta['dice_range'].values.astype(str),
    malignancy=meta['malignancy'].values.astype(str),
    dice_mean=meta['dice_score_mean'].values)

d = np.load(f'{P}/ima_256.npz')
print("Photos:", d['X'].shape, "| Outlines:", d['M'].shape)
print("Split counts:", pd.Series(d['split']).value_counts().to_dict())
print("Outlines per image:", pd.Series(d['K']).value_counts().sort_index().to_dict())
print("Outlines whose shape differs from their photo:", shape_mismatch)